In [1]:
import pandas as pd
import numpy as np

In [2]:
layoff = pd.read_csv("layoffs.csv")
layoff.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
0,VideoAmp,Los Angeles,50.0,8/7/2026,0.20,Marketing,https://www.wsj.com/cmo-today/videoamp-cuts-st...,Series F,456.0,United States,8/7/2026
1,TikTok,Nashville,250.0,8/6/2026,NaN,Consumer,https://www.nytimes.com/2026/08/05/technology/...,Acquired,NaN,United States,8/6/2026
2,Etsy,New York City,220.0,8/5/2026,0.12,Retail,https://www.cnbc.com/2026/08/05/etsy-layoffs-q...,Post-IPO,97.0,United States,8/5/2026
3,Google,SF Bay Area,52.0,8/5/2026,NaN,Consumer,https://www.geekwire.com/2026/google-to-cut-52...,Post-IPO,26.0,United States,8/6/2026
4,LegalZoom,Austin,NaN,8/5/2026,0.13,Legal,https://finance.yahoo.com/markets/stocks/artic...,Post-IPO,952.0,United States,8/7/2026


In [3]:
layoff.shape

(4554, 11)

In [4]:
layoff.info()

<class 'pandas.DataFrame'>
RangeIndex: 4554 entries, 0 to 4553
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   company              4554 non-null   str    
 1   location             4553 non-null   str    
 2   total_laid_off       2978 non-null   float64
 3   date                 4554 non-null   str    
 4   percentage_laid_off  2857 non-null   float64
 5   industry             4552 non-null   str    
 6   source               4551 non-null   str    
 7   stage                4546 non-null   str    
 8   funds_raised         4014 non-null   float64
 9   country              4552 non-null   str    
 10  date_added           4554 non-null   str    
dtypes: float64(3), str(8)
memory usage: 391.5 KB


In [5]:
layoff['date'] = pd.to_datetime(layoff['date'], errors='coerce')

In [6]:
layoff['date_added'] = pd.to_datetime(layoff['date_added'], errors='coerce')

In [7]:
layoff = layoff.drop_duplicates()

In [8]:
layoff.dtypes

company                           str
location                          str
total_laid_off                float64
date                   datetime64[us]
percentage_laid_off           float64
industry                          str
source                            str
stage                             str
funds_raised                  float64
country                           str
date_added             datetime64[us]
dtype: object

In [9]:
layoff['company'] = layoff['company'].str.strip()
layoff['industry'] = layoff['industry'].str.strip()

In [10]:
#layoff['total_laid_off '] = pd.to_numeric(layoff['total_laid_off'], errors='coerce')
layoff['percentage_laid_off'] = pd.to_numeric(layoff['percentage_laid_off'], errors='coerce')
layoff['funds_raised'] = pd.to_numeric(layoff['funds_raised'], errors='coerce')

In [11]:
layoff.isna().sum()

company                   0
location                  1
total_laid_off         1576
date                      0
percentage_laid_off    1697
industry                  2
source                    3
stage                     8
funds_raised            540
country                   2
date_added                0
dtype: int64

In [12]:
layoff[layoff['location'].isna()]

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
1441,Product Hunt,NaN,NaN,2023-10-09,0.6,Consumer,https://techcrunch.com/2023/10/19/product-hunt...,Acquired,NaN,United States,2023-10-15


In [15]:
layoff[layoff['total_laid_off'].isna()]

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
4,LegalZoom,Austin,NaN,2026-08-05,0.13,Legal,https://finance.yahoo.com/markets/stocks/artic...,Post-IPO,952.0,United States,2026-08-07
6,80 Acres Farms,Cincinnati,NaN,2026-08-04,1.00,Food,https://agfundernews.com/indoor-ag-heavyweight...,Unknown,275.0,United States,2026-08-04
7,jalebi.io,"Dubai, Non-U.S.",NaN,2026-08-04,1.00,Food,https://jawlah.co/en/61711,Seed,NaN,UAE,2026-08-04
8,Nutanix,SF Bay Area,NaN,2026-08-04,0.05,Infrastructure,https://www.crn.com/news/cloud/2026/nutanix-to...,Post-IPO,1100.0,United States,2026-08-04
9,FalconX,SF Bay Area,NaN,2026-08-03,0.10,Crypto,https://cryptobriefing.com/falconx-cuts-workfo...,Series D,477.0,United States,2026-08-03
...,...,...,...,...,...,...,...,...,...,...,...
4538,Vacasa,Portland,NaN,2020-03-20,NaN,Travel,https://www.bizjournals.com/portland/news/2020...,Series C,526.0,United States,2020-03-28
4543,Anyvision,"Tel Aviv, Non-U.S.",NaN,2020-03-19,NaN,Security,https://ipvm.com/reports/anyvision-20-layoffs,Series A,74.0,Israel,2020-03-30
4544,Popin,New York City,NaN,2020-03-19,1.00,Fitness,https://www.businessinsider.com/fitness-app-po...,Unknown,13.0,United States,2020-04-06
4545,Tuft & Needle,Phoenix,NaN,2020-03-19,NaN,Retail,https://www.theverge.com/2020/3/19/21185840/tu...,Acquired,NaN,United States,2020-04-05


In [16]:

both_known = layoff['percentage_laid_off'].notna() &  layoff['total_laid_off'].notna() & (layoff['percentage_laid_off'] > 0)
layoff.loc[both_known ,'implied_headcount'] = layoff.loc[both_known,'total_laid_off',] / layoff.loc[both_known,'percentage_laid_off'] * 100


In [17]:
# median headcount per industry (fallback to global median)
industry_headcount = layoff.groupby('industry')['implied_headcount'].median()
global_headcount = layoff['implied_headcount'].median()


In [18]:
def get_headcount(industry):
    val = industry_headcount.get(industry, np.nan)
    return val if pd.notna(val) else global_headcount

In [19]:

# 3a. percentage missing, total known -> derive percentage
mask = layoff['percentage_laid_off'].isna() & layoff['total_laid_off'].notna()
layoff.loc[mask, 'percentage_laid_off'] = layoff.loc[mask].apply(
    lambda r: min(r['total_laid_off'] / get_headcount(r['industry']), 1.0), axis=1
)

In [21]:
layoff.head()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added,implied_headcount
0,VideoAmp,Los Angeles,50.0,2026-08-07,0.200000,Marketing,https://www.wsj.com/cmo-today/videoamp-cuts-st...,Series F,456.0,United States,2026-08-07,25000.000000
1,TikTok,Nashville,250.0,2026-08-06,0.002250,Consumer,https://www.nytimes.com/2026/08/05/technology/...,Acquired,NaN,United States,2026-08-06,NaN
2,Etsy,New York City,220.0,2026-08-05,0.120000,Retail,https://www.cnbc.com/2026/08/05/etsy-layoffs-q...,Post-IPO,97.0,United States,2026-08-05,183333.333333
3,Google,SF Bay Area,52.0,2026-08-05,0.000468,Consumer,https://www.geekwire.com/2026/google-to-cut-52...,Post-IPO,26.0,United States,2026-08-06,NaN
4,LegalZoom,Austin,NaN,2026-08-05,0.130000,Legal,https://finance.yahoo.com/markets/stocks/artic...,Post-IPO,952.0,United States,2026-08-07,NaN


In [22]:
# 3b. total missing, percentage known -> derive total
mask = layoff['total_laid_off'].isna() & layoff['percentage_laid_off'].notna()
layoff.loc[mask, 'total_laid_off'] = layoff.loc[mask].apply(
    lambda r: round(r['percentage_laid_off'] * get_headcount(r['industry'])), axis=1
)

In [23]:
# 3c. both still missing -> fall back to group median (industry -> stage -> global)
for col in ['total_laid_off', 'percentage_laid_off']:
    layoff[col] = layoff[col].fillna(layoff.groupby('industry')[col].transform('median'))
    layoff[col] = layoff[col].fillna(layoff.groupby('stage')[col].transform('median'))
    layoff[col] = layoff[col].fillna(layoff[col].median())


In [24]:
layoff.drop(columns=['implied_headcount'], inplace=True, errors='ignore')

In [25]:
# ---------- 4. IMPUTE funds_raised ----------
# Funding is company/stage-specific -> use industry+stage median, fallback stage, fallback global
layoff['funds_raised'] = layoff['funds_raised'].fillna(
    layoff.groupby(['industry', 'stage'])['funds_raised'].transform('median')
)
layoff['funds_raised'] = layoff['funds_raised'].fillna(
    layoff.groupby('stage')['funds_raised'].transform('median')
)
layoff['funds_raised'] = layoff['funds_raised'].fillna(layoff['funds_raised'].median())

In [33]:
# ---------- 5. FINAL TOUCHES ----------
layoff['total_laid_off'] = layoff['total_laid_off'].round().astype('Int64')
layoff['percentage_laid_off'] = layoff['percentage_laid_off'].round(3)
layoff['funds_raised'] = layoff['funds_raised'].round(1)

# fill remaining small categorical/location gaps
for col in ['industry', 'stage', 'country', 'location', 'source']:
    layoff[col] = layoff[col].fillna('Not known')

print("Remaining nulls:\n", layoff.isna().sum())



Remaining nulls:
 company                0
location               0
total_laid_off         0
date                   0
percentage_laid_off    0
industry               0
source                 0
stage                  0
funds_raised           0
country                0
date_added             0
dtype: int64


In [34]:
layoff.to_csv('layoffs_cleaned.csv', index=False)

In [35]:
clean = pd.read_csv('layoffs_cleaned.csv')
clean.sample()

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
2986,LiveRamp,SF Bay Area,4333,2022-11-03,0.1,Marketing,https://www.marketscreener.com/quote/stock/LIV...,Post-IPO,16.0,United States,2022-11-05


In [36]:
# ---------- 6. QUICK ANALYSIS ----------
print("\n--- Top 10 industries by total layoffs ---")
print(clean.groupby('industry')['total_laid_off'].sum().sort_values(ascending=False).head(10))


--- Top 10 industries by total layoffs ---
industry
Hardware          7960705
Finance           2696162
Other             2603780
Consumer          2578274
Food              2477294
Retail            2325752
Transportation    2187525
Healthcare        1686700
Crypto             839042
Travel             747905
Name: total_laid_off, dtype: int64


In [37]:
print("\n--- Top 10 countries by total layoffs ---")
print(clean.groupby('country')['total_laid_off'].sum().sort_values(ascending=False).head(10))


--- Top 10 countries by total layoffs ---
country
United States     20244788
India              3016666
Israel             2909464
United Kingdom     1147591
Australia           657848
Canada              599709
Germany             533652
Nigeria             485911
Indonesia           359910
Singapore           284963
Name: total_laid_off, dtype: int64


In [38]:
print("\n--- Layoffs by company stage ---")
print(clean.groupby('stage')['total_laid_off'].sum().sort_values(ascending=False))


--- Layoffs by company stage ---
stage
Unknown           9314645
Series B          6122211
Seed              5258096
Post-IPO          3679668
Series A          2948020
Acquired          1689876
Series C          1654703
Series D           851674
Series E           338038
Series F           266412
Private Equity     222014
Subsidiary         152939
Not known          116750
Series H            64352
Series G            39920
Series J             4950
Series I             3255
Name: total_laid_off, dtype: int64


In [39]:
print("\n--- Monthly layoff trend ---")
monthly = clean.dropna(subset=['date']).groupby(layoff['date'].dt.to_period('M'))['total_laid_off'].sum()
print(monthly.tail(12))



--- Monthly layoff trend ---
date
2025-09    325209
2025-10     67807
2025-11    130656
2025-12    245551
2026-01    137141
2026-02     66660
2026-03    435231
2026-04    163795
2026-05    239699
2026-06    107232
2026-07    389954
2026-08    155511
Freq: M, Name: total_laid_off, dtype: int64
